## The EAR (Eye Aspect Ratio) Function File Codes

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os
import csv

mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh(static_image_mode=True, max_num_faces=1)

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [263, 387, 385, 362, 380, 373]

def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def compute_ear(landmarks, eye_indices):
    A = euclidean(landmarks[eye_indices[1]], landmarks[eye_indices[5]])
    B = euclidean(landmarks[eye_indices[2]], landmarks[eye_indices[4]])
    C = euclidean(landmarks[eye_indices[0]], landmarks[eye_indices[3]])
    return (A + B) / (2.0 * C) if C != 0 else 0

def process_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return None

    h, w, _ = image.shape
    results = face_mesh.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    if results.multi_face_landmarks:
        lm = results.multi_face_landmarks[0].landmark
        coords = [(int(pt.x * w), int(pt.y * h)) for pt in lm]

        left_ear = compute_ear(coords, LEFT_EYE)
        right_ear = compute_ear(coords, RIGHT_EYE)
        return (left_ear + right_ear) / 2.0
    return None

## The PERCLOS Batch Processing File Codes

In [ ]:
input_root = "dataset/NTHU-DDD/train_data"
output_csv = "eye_features.csv"
threshold = 0.21  # EAR threshold for eye closure

rows = []
for label_folder in ["drowsy", "notdrowsy"]:
    label = 1 if label_folder == "drowsy" else 0
    folder_path = os.path.join(input_root, label_folder)

    image_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.jpg')])
    closed_count = 0
    ear_values = []

    for filename in image_files:
        image_path = os.path.join(folder_path, filename)
        ear = process_image(image_path)
        if ear is not None:
            ear_values.append(ear)
            if ear < threshold:
                closed_count += 1
            rows.append([filename, ear, label])

    # PERCLOS for the folder
    total = len(ear_values)
    perclos = closed_count / total if total > 0 else 0
    print(f"{label_folder.upper()} — Images: {total}, Closed: {closed_count}, PERCLOS: {perclos:.2f}")

# Save to CSV
with open(output_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['image_name', 'ear', 'label'])
    writer.writerows(rows)

print(f"\n✅ Feature extraction complete. Saved to: {output_csv}")